## 1. Cài đặt thư viện

In [1]:
!pip install rouge-score bert-score sacrebleu nltk matplotlib pandas seaborn tqdm bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 1.1 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 2.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 5.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 3.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 10.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 4.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 3.7 MB/s eta 0:00:00a 0:00:01
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25000 sha256=bed27fdd0283f9d5c77d620bdc11cf7504f31b8f350aa4e2538408a431f3724e
  Stored in direc

In [3]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 620.1 kB/s eta 0:00:00:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.2 MB/s eta 0:00:00a 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 3.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: fsspec0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [propcache]
    Found existing installation: fsspec 2026.3.0━━━━━━━━━━━━━━  2/14 [propcache]
    Uninstalling fsspec-2026.3.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/14 [propcache]
      Successfully uninstalled fsspec-2026.3.0━━━━━━━━━━━━━━━━  2/14 [propcache]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [datasets]/14 [datasets]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Imports & Cấu hình

In [4]:
import os
import math
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_MODEL_DIR   = "./TinyLlama"
LORA_ADAPTER_DIR = "./LoRA/eye_llm_lora"
DATA_PATH        = "./eye_disease_train_v2.jsonl"
RESULTS_CSV      = "./eval_results_full.csv"

# ── Hyper-params ──────────────────────────────────────────────────────────────
RANDOM_SEED    = 42
TEST_RATIO     = 0.2
MAX_NEW_TOKENS = 256

# ── System prompt (giống project/model.py) ────────────────────────────────────
SYSTEM_PROMPT = (
    "You are EyeLlama, an AI assistant specialized in eye diseases and ophthalmology. "
    "You are not a human doctor and you are not any specific person. "
    "When asked who you are, always identify yourself as Eyesight. "
    "Provide helpful, accurate information about eye conditions, symptoms, diagnoses, and treatments."
)

# ── Style ─────────────────────────────────────────────────────────────────────
matplotlib.rcParams.update({"font.size": 12, "figure.dpi": 120})
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

/home/centrala/work/ou/kltn/EyeLlama/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
PyTorch: 2.8.0+cu128


## 3. Load & Tách tập dữ liệu

In [5]:
full_dataset = load_dataset("json", data_files=DATA_PATH)["train"]
split = full_dataset.train_test_split(test_size=TEST_RATIO, seed=RANDOM_SEED)
train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Tổng số mẫu  : {len(full_dataset)}")
print(f"Train        : {len(train_dataset)}")
print(f"Test         : {len(test_dataset)}")
print()
print("Ví dụ mẫu test:")
sample = test_dataset[0]
print(f"  instruction : {sample['instruction']}")
print(f"  input       : {sample['input']}")
print(f"  output      : {sample['output'][:120]}...")

Generating train split: 1012 examples [00:00, 173633.14 examples/s]

Tổng số mẫu  : 1012
Train        : 809
Test         : 203

Ví dụ mẫu test:
  instruction : Compare the injection burden of Myopic CNV vs AMD.
  input       : Anti-VEGF Frequency
  output      : Clinical data suggests that myopic CNV often requires fewer anti-VEGF injections to achieve inactivity compared to neova...


## 4. Load Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded:", BASE_MODEL_DIR)

Tokenizer loaded: ./TinyLlama


## 5. Hàm sinh câu trả lời & tính Perplexity

In [7]:
def build_prompt(sample: dict) -> str:
    """Tạo prompt theo chat template (giống lúc training)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"{sample['instruction']}\n{sample['input']}"}
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def generate_response(model, sample: dict, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    """Sinh câu trả lời bằng greedy decoding (deterministic)."""
    prompt = build_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy → reproducible
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Chỉ decode phần được sinh ra (bỏ prompt)
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def compute_perplexity(model, samples: list, batch_size: int = 4) -> float:
    """Tính Perplexity trung bình trên tập samples."""
    total_loss = 0.0
    total_tokens = 0

    for i in tqdm(range(0, len(samples), batch_size), desc="Perplexity"):
        batch = samples[i:i + batch_size]
        texts = []
        for s in batch:
            # Dùng full conversation (prompt + answer) để tính PPL
            messages = [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": f"{s['instruction']}\n{s['input']}"},
                {"role": "assistant", "content": s["output"]}
            ]
            texts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            ))

        enc = tokenizer(
            texts, return_tensors="pt",
            truncation=True, max_length=512,
            padding=True
        ).to(model.device)

        labels = enc["input_ids"].clone()
        # Mask padding tokens
        labels[labels == tokenizer.pad_token_id] = -100

        with torch.no_grad():
            loss = model(**enc, labels=labels).loss

        n_tokens = (labels != -100).sum().item()
        total_loss   += loss.item() * n_tokens
        total_tokens += n_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)


print("Inference functions defined.")

Inference functions defined.


## 6. Load Base Model & Sinh kết quả

In [8]:
print("[1/2] Loading Base TinyLlama...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.config.use_cache = True
base_model.eval()
print("Base model loaded.")

[1/2] Loading Base TinyLlama...
Base model loaded.


In [9]:
test_samples = [test_dataset[i] for i in range(len(test_dataset))]

print(f"Generating {len(test_samples)} responses from Base model...")
base_outputs = []
for s in tqdm(test_samples, desc="Base TinyLlama"):
    base_outputs.append(generate_response(base_model, s))

print("Done. Sample output:")
print(base_outputs[0][:200])

Generating 203 responses from Base model...


Base TinyLlama:   1%|          | 2/203 [04:46<7:59:51, 143.24s/it]


KeyboardInterrupt: 

In [ ]:
print("Computing perplexity for Base model...")
base_ppl = compute_perplexity(base_model, test_samples)
print(f"Base Perplexity: {base_ppl:.4f}")

In [ ]:
# Giải phóng VRAM trước khi load fine-tuned model
del base_model
torch.cuda.empty_cache()
print("Base model freed from GPU memory.")

## 7. Load Fine-tuned Model (EyeLlama) & Sinh kết quả

In [ ]:
print("[2/2] Loading EyeLlama (Base + LoRA adapter)...")

bnb_config2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config2,
    device_map="auto",
)
_base.config.use_cache = True

finetuned_model = PeftModel.from_pretrained(_base, LORA_ADAPTER_DIR)
finetuned_model.eval()
print("EyeLlama loaded.")

In [ ]:
print(f"Generating {len(test_samples)} responses from EyeLlama...")
finetuned_outputs = []
for s in tqdm(test_samples, desc="EyeLlama"):
    finetuned_outputs.append(generate_response(finetuned_model, s))

print("Done. Sample output:")
print(finetuned_outputs[0][:200])

In [ ]:
print("Computing perplexity for EyeLlama...")
finetuned_ppl = compute_perplexity(finetuned_model, test_samples)
print(f"EyeLlama Perplexity: {finetuned_ppl:.4f}")

In [ ]:
references = [s["output"] for s in test_samples]

df_results = pd.DataFrame({
    "instruction"      : [s["instruction"] for s in test_samples],
    "input"            : [s["input"]       for s in test_samples],
    "reference"        : references,
    "base_output"      : base_outputs,
    "finetuned_output" : finetuned_outputs,
})
df_results.to_csv(RESULTS_CSV, index=False)
print(f"Results saved to {RESULTS_CSV}  ({len(df_results)} rows)")
df_results.head(3)

## 8. BLEU Score

In [ ]:
import sacrebleu

def compute_bleu(predictions: list, references: list) -> dict:
    """Tính BLEU-1, BLEU-2, BLEU-4 dùng sacrebleu."""
    results = {}
    for n in [1, 2, 4]:
        bleu = sacrebleu.corpus_bleu(
            predictions,
            [references],
            max_ngram_order=n,
            smooth_method="exp"
        )
        results[f"BLEU-{n}"] = round(bleu.score, 4)
    return results

bleu_base      = compute_bleu(base_outputs,      references)
bleu_finetuned = compute_bleu(finetuned_outputs, references)

print("BLEU Scores")
print(f"{'Metric':<10} {'Base':>10} {'EyeLlama':>12}")
print("-" * 35)
for k in bleu_base:
    print(f"{k:<10} {bleu_base[k]:>10.4f} {bleu_finetuned[k]:>12.4f}")

## 9. ROUGE Scores

In [ ]:
from rouge_score import rouge_scorer

def compute_rouge(predictions: list, references: list) -> dict:
    """Tính ROUGE-1, ROUGE-2, ROUGE-L (F1 trung bình)."""
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for pred, ref in zip(predictions, references):
        result = scorer.score(ref, pred)
        for key in scores:
            scores[key].append(result[key].fmeasure)

    return {k: round(float(np.mean(v)), 4) for k, v in scores.items()}


def compute_rouge_per_sample(predictions: list, references: list) -> dict:
    """Trả về list điểm ROUGE-L từng mẫu (dùng cho box plot)."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    return [scorer.score(ref, pred)["rougeL"].fmeasure
            for pred, ref in zip(predictions, references)]


rouge_base      = compute_rouge(base_outputs,      references)
rouge_finetuned = compute_rouge(finetuned_outputs, references)
rougeL_base_list      = compute_rouge_per_sample(base_outputs,      references)
rougeL_finetuned_list = compute_rouge_per_sample(finetuned_outputs, references)

print("ROUGE Scores (F1)")
print(f"{'Metric':<10} {'Base':>10} {'EyeLlama':>12}")
print("-" * 35)
for k in rouge_base:
    label = k.upper().replace("ROUGE", "ROUGE-").replace("1","1").replace("2","2").replace("L","L")
    print(f"{label:<10} {rouge_base[k]:>10.4f} {rouge_finetuned[k]:>12.4f}")

## 10. BERTScore

In [ ]:
from bert_score import score as bert_score_fn

def compute_bertscore(predictions: list, references: list, lang: str = "en") -> dict:
    """Tính BERTScore Precision, Recall, F1 trung bình."""
    P, R, F1 = bert_score_fn(
        predictions, references,
        lang=lang,
        verbose=False,
        batch_size=8,
    )
    return {
        "BERTScore-P" : round(P.mean().item(),  4),
        "BERTScore-R" : round(R.mean().item(),  4),
        "BERTScore-F1": round(F1.mean().item(), 4),
    }

print("Computing BERTScore for Base model...")
bert_base = compute_bertscore(base_outputs, references)

print("Computing BERTScore for EyeLlama...")
bert_finetuned = compute_bertscore(finetuned_outputs, references)

print("\nBERTScore")
print(f"{'Metric':<14} {'Base':>10} {'EyeLlama':>12}")
print("-" * 39)
for k in bert_base:
    print(f"{k:<14} {bert_base[k]:>10.4f} {bert_finetuned[k]:>12.4f}")

## 11. Bảng tổng hợp tất cả Metrics

In [ ]:
summary = {
    "Metric": [
        "BLEU-1", "BLEU-2", "BLEU-4",
        "ROUGE-1", "ROUGE-2", "ROUGE-L",
        "BERTScore-P", "BERTScore-R", "BERTScore-F1",
        "Perplexity",
    ],
    "TinyLlama (Base)": [
        bleu_base["BLEU-1"], bleu_base["BLEU-2"], bleu_base["BLEU-4"],
        rouge_base["rouge1"], rouge_base["rouge2"], rouge_base["rougeL"],
        bert_base["BERTScore-P"], bert_base["BERTScore-R"], bert_base["BERTScore-F1"],
        round(base_ppl, 4),
    ],
    "EyeLlama (Fine-tuned)": [
        bleu_finetuned["BLEU-1"], bleu_finetuned["BLEU-2"], bleu_finetuned["BLEU-4"],
        rouge_finetuned["rouge1"], rouge_finetuned["rouge2"], rouge_finetuned["rougeL"],
        bert_finetuned["BERTScore-P"], bert_finetuned["BERTScore-R"], bert_finetuned["BERTScore-F1"],
        round(finetuned_ppl, 4),
    ],
}

df_summary = pd.DataFrame(summary)

# Tính % thay đổi (Perplexity: thấp hơn = tốt hơn; các metric khác: cao hơn = tốt hơn)
def pct_change(base_val, ft_val, lower_is_better=False):
    delta = ft_val - base_val
    pct = (delta / abs(base_val)) * 100 if base_val != 0 else 0
    if lower_is_better:
        pct = -pct
    return f"{'+' if pct >= 0 else ''}{pct:.2f}%"

pcts = []
for i, m in enumerate(summary["Metric"]):
    lb = (m == "Perplexity")
    pcts.append(pct_change(summary["TinyLlama (Base)"][i],
                           summary["EyeLlama (Fine-tuned)"][i],
                           lower_is_better=lb))

df_summary["Cải thiện"] = pcts

print("=" * 65)
print("        KẾT QUẢ THỰC NGHIỆM — SO SÁNH METRICS")
print("=" * 65)
print(df_summary.to_string(index=False))
print("=" * 65)

df_summary.to_csv("./eval_summary.csv", index=False)
print("Summary saved to ./eval_summary.csv")

## 12. Visualization

### 12.1 Biểu đồ BLEU & ROUGE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── BLEU ─────────────────────────────────────────────────────────────────────
bleu_labels  = ["BLEU-1", "BLEU-2", "BLEU-4"]
bleu_base_v  = [bleu_base[k]      for k in bleu_labels]
bleu_ft_v    = [bleu_finetuned[k] for k in bleu_labels]

x = np.arange(len(bleu_labels))
w = 0.35
ax = axes[0]
bars1 = ax.bar(x - w/2, bleu_base_v, w, label="TinyLlama (Base)",   color="#5b9bd5", alpha=0.85)
bars2 = ax.bar(x + w/2, bleu_ft_v,   w, label="EyeLlama (Fine-tuned)", color="#ed7d31", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(bleu_labels)
ax.set_ylabel("Score")
ax.set_title("BLEU Score", fontweight="bold")
ax.legend()
for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# ── ROUGE ─────────────────────────────────────────────────────────────────────
rouge_labels = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
rouge_base_v = [rouge_base["rouge1"], rouge_base["rouge2"], rouge_base["rougeL"]]
rouge_ft_v   = [rouge_finetuned["rouge1"], rouge_finetuned["rouge2"], rouge_finetuned["rougeL"]]

ax2 = axes[1]
x2 = np.arange(len(rouge_labels))
bars3 = ax2.bar(x2 - w/2, rouge_base_v, w, label="TinyLlama (Base)",   color="#5b9bd5", alpha=0.85)
bars4 = ax2.bar(x2 + w/2, rouge_ft_v,   w, label="EyeLlama (Fine-tuned)", color="#ed7d31", alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(rouge_labels)
ax2.set_ylabel("F1 Score")
ax2.set_title("ROUGE Score (F1)", fontweight="bold")
ax2.legend()
for bar in bars3: ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars4: ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002, f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("./fig_bleu_rouge.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_bleu_rouge.png")

### 12.2 Biểu đồ BERTScore

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bert_labels = ["Precision", "Recall", "F1"]
bert_base_v = [bert_base["BERTScore-P"], bert_base["BERTScore-R"], bert_base["BERTScore-F1"]]
bert_ft_v   = [bert_finetuned["BERTScore-P"], bert_finetuned["BERTScore-R"], bert_finetuned["BERTScore-F1"]]

x = np.arange(len(bert_labels))
w = 0.35
bars1 = ax.bar(x - w/2, bert_base_v, w, label="TinyLlama (Base)",   color="#5b9bd5", alpha=0.85)
bars2 = ax.bar(x + w/2, bert_ft_v,   w, label="EyeLlama (Fine-tuned)", color="#ed7d31", alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(bert_labels)
ax.set_ylabel("Score")
ax.set_ylim(0.8, 1.0)   # BERTScore thường cao > 0.8
ax.set_title("BERTScore", fontweight="bold")
ax.legend()

for bar in bars1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)
for bar in bars2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001, f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("./fig_bertscore.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_bertscore.png")

### 12.3 Biểu đồ Perplexity

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

models = ["TinyLlama\n(Base)", "EyeLlama\n(Fine-tuned)"]
ppls   = [base_ppl, finetuned_ppl]
colors = ["#5b9bd5", "#ed7d31"]

bars = ax.bar(models, ppls, color=colors, alpha=0.85, width=0.5)
for bar, val in zip(bars, ppls):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_ylabel("Perplexity (thấp hơn = tốt hơn)")
ax.set_title("Perplexity trên tập Test", fontweight="bold")
ax.set_ylim(0, max(ppls) * 1.25)

plt.tight_layout()
plt.savefig("./fig_perplexity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_perplexity.png")

### 12.4 Box Plot phân phối ROUGE-L từng mẫu

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

df_box = pd.DataFrame({
    "TinyLlama (Base)"     : rougeL_base_list,
    "EyeLlama (Fine-tuned)": rougeL_finetuned_list,
})

df_box.boxplot(ax=ax, patch_artist=True,
               boxprops=dict(facecolor="lightblue"),
               medianprops=dict(color="red", linewidth=2))

ax.set_ylabel("ROUGE-L F1")
ax.set_title("Phân phối ROUGE-L theo từng mẫu test", fontweight="bold")

plt.tight_layout()
plt.savefig("./fig_rougeL_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: fig_rougeL_distribution.png")

## 13. Ví dụ định tính (Qualitative Examples)

In [ ]:
NUM_EXAMPLES = 5
# Chọn mẫu cố định để dễ trình bày trong khóa luận
example_indices = list(range(NUM_EXAMPLES))

print("=" * 80)
print("VÍ DỤ ĐỊNH TÍNH: SO SÁNH BASE vs FINE-TUNED")
print("=" * 80)

for idx in example_indices:
    s = test_samples[idx]
    print(f"\n[Mẫu {idx+1}]")
    print(f"Input      : {s['instruction']} | {s['input']}")
    print(f"Reference  : {s['output']}")
    print(f"Base       : {base_outputs[idx]}")
    print(f"EyeLlama   : {finetuned_outputs[idx]}")
    print("-" * 80)

In [ ]:
# Hiển thị dạng bảng (dễ screenshot cho khóa luận)
df_qual = df_results.iloc[example_indices][["instruction", "input", "reference", "base_output", "finetuned_output"]].copy()
df_qual.columns = ["Instruction", "Input", "Reference", "TinyLlama Base", "EyeLlama"]
df_qual.index = [f"Mẫu {i+1}" for i in range(NUM_EXAMPLES)]
df_qual

## 14. Kết luận tổng hợp

In [ ]:
print("\n" + "═" * 65)
print("  KẾT QUẢ THỰC NGHIỆM — TÓM TẮT")
print("═" * 65)
print(f"  Mô hình Base    : TinyLlama-1.1B-Chat-v1.0")
print(f"  Mô hình FT      : EyeLlama (LoRA r=8, α=16, epoch=3)")
print(f"  Tập test        : {len(test_samples)} mẫu (20% của {len(full_dataset)})")
print(f"  Decoding        : Greedy (do_sample=False)")
print(f"  Max new tokens  : {MAX_NEW_TOKENS}")
print("─" * 65)
print(f"  {'Metric':<18} {'Base':>12} {'EyeLlama':>14} {'Δ':>8}")
print("─" * 65)

metric_rows = [
    ("BLEU-1",       bleu_base["BLEU-1"],      bleu_finetuned["BLEU-1"],      False),
    ("BLEU-2",       bleu_base["BLEU-2"],      bleu_finetuned["BLEU-2"],      False),
    ("BLEU-4",       bleu_base["BLEU-4"],      bleu_finetuned["BLEU-4"],      False),
    ("ROUGE-1 (F1)", rouge_base["rouge1"],     rouge_finetuned["rouge1"],     False),
    ("ROUGE-2 (F1)", rouge_base["rouge2"],     rouge_finetuned["rouge2"],     False),
    ("ROUGE-L (F1)", rouge_base["rougeL"],     rouge_finetuned["rougeL"],     False),
    ("BERTScore-F1", bert_base["BERTScore-F1"], bert_finetuned["BERTScore-F1"], False),
    ("Perplexity",   base_ppl,                  finetuned_ppl,                  True),
]

for name, bv, fv, lower_better in metric_rows:
    delta = fv - bv
    sign  = "+" if delta >= 0 else ""
    improved = (delta < 0) if lower_better else (delta >= 0)
    marker = "✓" if improved else "✗"
    print(f"  {name:<18} {bv:>12.4f} {fv:>14.4f} {sign}{delta:>6.4f} {marker}")

print("═" * 65)
print("  ✓ = EyeLlama cải thiện so với Base")
print("  Perplexity: thấp hơn = tốt hơn; các metric còn lại: cao hơn = tốt hơn")
print("═" * 65)